# LeetCode 594 — Longest Harmonious Subsequence bằng Python

**Link đề:** https://leetcode.com/problems/longest-harmonious-subsequence/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sliding Window, Sorting, Counting

```python
class Solution:
    def findLHS(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Phân biệt **subsequence** (dãy con, giữ thứ tự, không cần liền nhau) với **subarray** (đoạn con liền nhau) – và nhận ra: với bài này **thứ tự không quan trọng**, chỉ **số lần xuất hiện** quan trọng.
2. Đi từ brute force (thử mọi dãy con `O(2^n)`) → đếm từng cặp `O(n²)` → **sắp xếp + cửa sổ trượt** `O(n log n)` → **đếm tần suất bằng `Counter`** `O(n)`.
3. Thành thạo `dict`, `collections.Counter`, `sorted`, hai con trỏ.
4. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, làm 6 bài tập có đáp án ở cuối.

## 1. Đề bài và ví dụ

Một mảng gọi là **harmonious** (hài hòa) nếu **max − min đúng bằng 1**.

Cho mảng số nguyên `nums`, trả về **độ dài lớn nhất** của một **dãy con (subsequence)** hài hòa.

| nums | Output | Giải thích |
|---|---:|---|
| `[1,3,2,2,5,2,3,7]` | `5` | chọn `[3,2,2,2,3]`: max 3, min 2 |
| `[1,2,3,4]` | `2` | `[1,2]`, `[2,3]`, `[3,4]` đều dài 2 |
| `[1,1,1,1]` | `0` | max − min = 0 ≠ 1 → không có dãy hài hòa |

**Constraints:** `1 <= nums.length <= 2 * 10^4`, `-10^9 <= nums[i] <= 10^9`.

Chú ý: "**đúng bằng** 1" – dãy toàn số giống nhau (chênh 0) **không** tính. Vì vậy `[1,1,1,1]` ra `0`.

## 2. Nền tảng: subsequence là gì? Vì sao chỉ cần đếm?

**Subsequence** = xóa đi 0 hoặc nhiều phần tử, giữ nguyên thứ tự phần còn lại. Ví dụ từ `[1,3,2,2,5,2,3,7]` có thể lấy `[3,2,2,2,3]` (vị trí 1,2,3,5,6) – không cần liền nhau.

Quan sát quan trọng:
1. Dãy hài hòa có max − min = 1 → chỉ chứa **đúng hai giá trị** `x` và `x+1` (và phải có **cả hai**).
2. Vì được bỏ phần tử tùy ý, muốn dài nhất thì **lấy hết** mọi `x` và mọi `x+1` trong mảng.
3. Thứ tự giữ nguyên tự động (ta chỉ xóa phần tử khác) → **không cần quan tâm vị trí**.

⇒ Đáp án = max của `count[x] + count[x+1]` trên mọi `x` mà **cả `x` lẫn `x+1` đều xuất hiện**.

Thử bằng tay với ví dụ 1: đếm `1:1, 2:3, 3:2, 5:1, 7:1`.
- `x=1`: có 2 → `1 + 3 = 4`
- `x=2`: có 3 → `3 + 2 = 5`  ← lớn nhất
- `x=3`: không có 4 → bỏ; `x=5`: không có 6 → bỏ; `x=7`: không có 8 → bỏ.

In [1]:
nums = [1, 3, 2, 2, 5, 2, 3, 7]
picked = [v for v in nums if v in (2, 3)]   # giữ nguyên thứ tự, chỉ lấy 2 và 3
print("Dãy con chỉ gồm 2 và 3:", picked)
print("max - min =", max(picked) - min(picked), "| độ dài =", len(picked))

Dãy con chỉ gồm 2 và 3: [3, 2, 2, 2, 3]
max - min = 1 | độ dài = 5


### 2.1. `dict` để đếm tần suất

`dict` lưu cặp **key → value**. Để đếm: key là giá trị, value là số lần gặp.
`d.get(k, 0)` trả về `d[k]` nếu có, ngược lại trả `0` (không lỗi `KeyError`).

In [2]:
count = {}
for v in [1, 3, 2, 2, 5, 2, 3, 7]:
    count[v] = count.get(v, 0) + 1
print(count)
print("Có số 4 không?", 4 in count, "| count.get(4, 0) =", count.get(4, 0))

{1: 1, 3: 2, 2: 3, 5: 1, 7: 1}
Có số 4 không? False | count.get(4, 0) = 0


### 2.2. `collections.Counter` – dict chuyên để đếm

`Counter(iterable)` đếm sẵn cho ta. Truy cập key không tồn tại trả về `0` (không lỗi). Nhưng lưu ý: `c[k]` với key mới **không** thêm key vào Counter, còn `k in c` mới là cách kiểm tra "có xuất hiện".

In [3]:
from collections import Counter
c = Counter([1, 3, 2, 2, 5, 2, 3, 7])
print(c)
print("c[2] =", c[2], "| c[4] =", c[4], "| 4 in c ?", 4 in c)
print("Duyệt key-value:", [(k, v) for k, v in c.items()])

Counter({2: 3, 3: 2, 1: 1, 5: 1, 7: 1})
c[2] = 3 | c[4] = 0 | 4 in c ? False
Duyệt key-value: [(1, 1), (3, 2), (2, 3), (5, 1), (7, 1)]


## 3. Cách 0 — Brute force thật sự: thử mọi dãy con (chỉ để làm "oracle")

Mỗi phần tử có 2 lựa chọn: lấy / không lấy → `2^n` dãy con. Ta dùng **bitmask**: số `mask` từ `0` tới `2^n − 1`; bit thứ `i` bằng 1 nghĩa là lấy `nums[i]`.

`mask >> i & 1` lấy bit thứ `i` của `mask`.

Cách này **quá chậm** (`n = 20000` → không thể), nhưng rất hữu ích để **kiểm tra** lời giải nhanh trên mảng nhỏ.

In [4]:
mask = 0b101          # = 5: lấy phần tử 0 và 2
arr = ["a", "b", "c"]
print([arr[i] for i in range(3) if mask >> i & 1])

['a', 'c']


In [5]:
def lhs_bruteforce(nums):
    n = len(nums)
    best = 0
    for mask in range(1, 1 << n):          # 1 << n = 2**n
        sub = [nums[i] for i in range(n) if mask >> i & 1]
        if max(sub) - min(sub) == 1:
            best = max(best, len(sub))
    return best

print(lhs_bruteforce([1, 3, 2, 2, 5, 2, 3, 7]))  # 5
print(lhs_bruteforce([1, 2, 3, 4]))              # 2
print(lhs_bruteforce([1, 1, 1, 1]))              # 0

5
2
0


## 4. Cách 1 — Với mỗi `x`, quét mảng đếm `x` và `x+1` (`O(n²)`)

Dùng quan sát ở mục 2 nhưng chưa biết `dict`: với mỗi giá trị `x = nums[i]`, quét lại cả mảng đếm số phần tử bằng `x` và bằng `x+1`. Chỉ nhận nếu số `x+1` > 0.

In [6]:
def lhs_quadratic(nums):
    best = 0
    for x in nums:
        cnt_x = cnt_next = 0
        for y in nums:
            if y == x:
                cnt_x += 1
            elif y == x + 1:
                cnt_next += 1
        if cnt_next > 0:                  # phải có cả x+1
            best = max(best, cnt_x + cnt_next)
    return best

print(lhs_quadratic([1, 3, 2, 2, 5, 2, 3, 7]))
print(lhs_quadratic([1, 2, 3, 4]))
print(lhs_quadratic([1, 1, 1, 1]))

5
2
0


**Nút thắt:** với `n = 2·10^4`, `n² = 4·10^8` phép so sánh – Python quá chậm. Công việc lặp lại: ta **đếm đi đếm lại** cùng một giá trị nhiều lần. → Hoặc **sắp xếp** để các giá trị bằng nhau đứng cạnh nhau, hoặc **đếm một lần** rồi tra cứu.

## 5. Cách 2 — Sắp xếp + cửa sổ trượt (hai con trỏ), `O(n log n)`

### 5.1. Ý tưởng
Sau khi sắp xếp, mọi số `x` và `x+1` nằm thành **một đoạn liền nhau**. Ta giữ cửa sổ `[left, right]` sao cho `nums[right] − nums[left] <= 1`:
- Mở rộng `right` từng bước.
- Nếu chênh lệch > 1 → tăng `left` cho tới khi hợp lệ lại.
- Nếu chênh lệch **đúng bằng 1** → cửa sổ là một dãy hài hòa, cập nhật `best = right − left + 1`.

### 5.2. `sorted` vs `.sort()`
`sorted(lst)` trả về list **mới**; `lst.sort()` sắp xếp **tại chỗ** và trả `None`. Ta dùng `sorted` để không làm đổi input của người gọi.

In [7]:
data = [1, 3, 2, 2, 5, 2, 3, 7]
print("sorted(data):", sorted(data))
print("data vẫn giữ nguyên:", data)

sorted(data): [1, 2, 2, 2, 3, 3, 5, 7]
data vẫn giữ nguyên: [1, 3, 2, 2, 5, 2, 3, 7]


In [8]:
def lhs_sort_window(nums):
    arr = sorted(nums)
    best = 0
    left = 0
    for right in range(len(arr)):
        while arr[right] - arr[left] > 1:     # cửa sổ quá rộng -> thu hẹp
            left += 1
        if arr[right] - arr[left] == 1:       # đúng hài hòa
            best = max(best, right - left + 1)
    return best

print(lhs_sort_window([1, 3, 2, 2, 5, 2, 3, 7]))
print(lhs_sort_window([1, 2, 3, 4]))
print(lhs_sort_window([1, 1, 1, 1]))

5
2
0


### 5.3. TRACE cửa sổ trượt
`arr = [1, 2, 2, 2, 3, 3, 5, 7]`

| right | arr[right] | left sau thu hẹp | cửa sổ | chênh | best |
|---:|---:|---:|---|---:|---:|
| 0 | 1 | 0 | `[1]` | 0 | 0 |
| 1 | 2 | 0 | `[1,2]` | 1 | 2 |
| 2 | 2 | 0 | `[1,2,2]` | 1 | 3 |
| 3 | 2 | 0 | `[1,2,2,2]` | 1 | 4 |
| 4 | 3 | 1 | `[2,2,2,3]` | 1 | 4 |
| 5 | 3 | 1 | `[2,2,2,3,3]` | 1 | **5** |
| 6 | 5 | 6 | `[5]` | 0 | 5 |
| 7 | 7 | 7 | `[7]` | 0 | 5 |

In [9]:
def trace_sort_window(nums):
    arr = sorted(nums)
    print("arr =", arr)
    best = left = 0
    for right in range(len(arr)):
        while arr[right] - arr[left] > 1:
            left += 1
        diff = arr[right] - arr[left]
        if diff == 1:
            best = max(best, right - left + 1)
        print(f"right={right} val={arr[right]} | left={left} | cửa sổ={arr[left:right+1]} | chênh={diff} | best={best}")
    return best

trace_sort_window([1, 3, 2, 2, 5, 2, 3, 7])

arr = [1, 2, 2, 2, 3, 3, 5, 7]
right=0 val=1 | left=0 | cửa sổ=[1] | chênh=0 | best=0
right=1 val=2 | left=0 | cửa sổ=[1, 2] | chênh=1 | best=2
right=2 val=2 | left=0 | cửa sổ=[1, 2, 2] | chênh=1 | best=3
right=3 val=2 | left=0 | cửa sổ=[1, 2, 2, 2] | chênh=1 | best=4
right=4 val=3 | left=1 | cửa sổ=[2, 2, 2, 3] | chênh=1 | best=4
right=5 val=3 | left=1 | cửa sổ=[2, 2, 2, 3, 3] | chênh=1 | best=5
right=6 val=5 | left=6 | cửa sổ=[5] | chênh=0 | best=5
right=7 val=7 | left=7 | cửa sổ=[7] | chênh=0 | best=5


5

## 6. Insight chính: đếm một lần, tra cứu `x+1` trong `O(1)`

Từ mục 2: đáp án = `max(count[x] + count[x+1])` với `x+1` có mặt. Đếm bằng `Counter` mất `O(n)`; với mỗi key `x` (tối đa `n` key) kiểm tra `x + 1 in count` trung bình `O(1)`.

Giảng từng đoạn:
1. `count = Counter(nums)` – bảng tần suất.
2. `for x in count:` – duyệt các **giá trị khác nhau** (không duyệt trùng).
3. `if x + 1 in count:` – chỉ khi có "bạn đôi" `x+1`.
4. `best = max(best, count[x] + count[x + 1])`.

In [10]:
def lhs_counter(nums):
    count = Counter(nums)
    best = 0
    for x in count:
        if x + 1 in count:
            best = max(best, count[x] + count[x + 1])
    return best

print(lhs_counter([1, 3, 2, 2, 5, 2, 3, 7]))
print(lhs_counter([1, 2, 3, 4]))
print(lhs_counter([1, 1, 1, 1]))

5
2
0


### 6.1. TRACE cách Counter

| x | count[x] | x+1 có không? | count[x+1] | tổng | best |
|---:|---:|:---:|---:|---:|---:|
| 1 | 1 | có (2) | 3 | 4 | 4 |
| 3 | 2 | không (4) | – | – | 4 |
| 2 | 3 | có (3) | 2 | 5 | **5** |
| 5 | 1 | không | – | – | 5 |
| 7 | 1 | không | – | – | 5 |

(Thứ tự duyệt key của `Counter` là thứ tự **xuất hiện lần đầu** trong mảng.)

In [11]:
def trace_counter(nums):
    count = Counter(nums)
    print("count =", dict(count))
    best = 0
    for x in count:
        if x + 1 in count:
            total = count[x] + count[x + 1]
            best = max(best, total)
            print(f"x={x}: count[x]={count[x]}, có {x+1} (count={count[x+1]}) -> tổng {total}, best={best}")
        else:
            print(f"x={x}: không có {x+1} -> bỏ qua")
    return best

trace_counter([1, 3, 2, 2, 5, 2, 3, 7])

count = {1: 1, 3: 2, 2: 3, 5: 1, 7: 1}
x=1: count[x]=1, có 2 (count=3) -> tổng 4, best=4
x=3: không có 4 -> bỏ qua
x=2: count[x]=3, có 3 (count=2) -> tổng 5, best=5
x=5: không có 6 -> bỏ qua
x=7: không có 8 -> bỏ qua


5

### 6.2. Biến thể một lượt (one pass)

Có thể vừa đếm vừa cập nhật: khi thêm `x`, nhìn cả `x−1` và `x+1` đã có. Kết quả giống hệt, chỉ một vòng lặp.

In [12]:
def lhs_one_pass(nums):
    count = {}
    best = 0
    for x in nums:
        count[x] = count.get(x, 0) + 1
        if x - 1 in count:
            best = max(best, count[x] + count[x - 1])
        if x + 1 in count:
            best = max(best, count[x] + count[x + 1])
    return best

print(lhs_one_pass([1, 3, 2, 2, 5, 2, 3, 7]), lhs_one_pass([1, 2, 3, 4]), lhs_one_pass([1, 1, 1, 1]))

5 2 0


## 7. Code nộp LeetCode

Bản chính (Counter):

In [13]:
from collections import Counter

class Solution:
    def findLHS(self, nums: list[int]) -> int:
        count = Counter(nums)
        best = 0
        for x in count:
            if x + 1 in count:
                best = max(best, count[x] + count[x + 1])
        return best


sol = Solution()
assert sol.findLHS([1, 3, 2, 2, 5, 2, 3, 7]) == 5
assert sol.findLHS([1, 2, 3, 4]) == 2
assert sol.findLHS([1, 1, 1, 1]) == 0
print("Solution (Counter) PASS")

Solution (Counter) PASS


Bản sắp xếp + cửa sổ trượt (dùng khi không muốn tốn bộ nhớ hash, hoặc để luyện two pointers):

In [14]:
class SolutionSortWindow:
    def findLHS(self, nums: list[int]) -> int:
        nums = sorted(nums)
        best = left = 0
        for right in range(len(nums)):
            while nums[right] - nums[left] > 1:
                left += 1
            if nums[right] - nums[left] == 1:
                best = max(best, right - left + 1)
        return best

assert SolutionSortWindow().findLHS([1, 3, 2, 2, 5, 2, 3, 7]) == 5
print("SolutionSortWindow PASS")

SolutionSortWindow PASS


## 8. Vì sao đúng?

**Bổ đề 1.** Mọi dãy hài hòa chứa đúng hai giá trị `x`, `x+1` (với `x` = min), có ít nhất một phần tử mỗi loại. Vì max − min = 1 và các số nguyên nằm giữa min, max chỉ có thể là min hoặc max.

**Bổ đề 2.** Với `x` cố định mà cả `x` và `x+1` xuất hiện, dãy con lấy **toàn bộ** các phần tử bằng `x` hoặc `x+1` (giữ thứ tự gốc) là hài hòa và dài nhất với cặp đó: dài `count[x] + count[x+1]`.

⇒ Đáp án = max trên mọi `x` hợp lệ; nếu không có `x` nào → 0. Cách Counter duyệt đúng tập `x` đó nên đúng.

**Cửa sổ trượt (invariant):** sau vòng `while`, `[left, right]` là **cửa sổ dài nhất kết thúc tại `right`** có chênh lệch ≤ 1 (vì `left` chỉ tăng khi bắt buộc). Nếu chênh = 1, cửa sổ đó gồm mọi `x` và `x+1` nhỏ hơn hoặc bằng vị trí `right`; tại `right` cuối cùng của khối `x+1`, ta có đúng `count[x] + count[x+1]`.

## 9. Time / Space Complexity và so sánh

`n` = độ dài `nums`, `k` = số giá trị khác nhau (`k <= n`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 0. Thử mọi dãy con | `O(2^n · n)` | `O(n)` | Chỉ làm oracle cho `n` nhỏ |
| 1. Quét đếm từng `x` | `O(n²)` | `O(1)` | Quá chậm với `n = 2·10^4` |
| 2. Sắp xếp + cửa sổ | `O(n log n)` | `O(n)` cho `sorted` (hoặc `O(1)` nếu sort tại chỗ) | Không dùng hash |
| 3. Counter | `O(n)` trung bình | `O(k)` | Bản chính, ngắn gọn |
| 3b. Một lượt | `O(n)` trung bình | `O(k)` | Giống 3, một vòng lặp |

"Trung bình" vì tra cứu `dict` là `O(1)` trung bình (hash), không phải tuyệt đối mọi trường hợp.

## 10. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu ngẫu nhiên

In [15]:
import random

methods = {
    "quadratic": lhs_quadratic,
    "sort_window": lhs_sort_window,
    "counter": lhs_counter,
    "one_pass": lhs_one_pass,
}
cases = [
    ("đề 1", [1, 3, 2, 2, 5, 2, 3, 7], 5),
    ("đề 2", [1, 2, 3, 4], 2),
    ("đề 3 (toàn giống nhau)", [1, 1, 1, 1], 0),
    ("1 phần tử", [5], 0),
    ("số âm", [-1, 0, -1, 0, 0], 5),
    ("chênh 2 không tính", [1, 3, 5, 7], 0),
    ("số rất lớn", [10**9, 10**9 - 1, -10**9], 2),
    ("nhiều cặp", [1, 1, 2, 2, 2, 3, 3, 3, 3], 7),
]
for name, nums, expected in cases:
    for mname, f in methods.items():
        got = f(nums)
        assert got == expected, (name, mname, got)
    print(f"{name:<24} PASS -> {expected}")

đề 1                     PASS -> 5
đề 2                     PASS -> 2
đề 3 (toàn giống nhau)   PASS -> 0
1 phần tử                PASS -> 0
số âm                    PASS -> 5
chênh 2 không tính       PASS -> 0
số rất lớn               PASS -> 2
nhiều cặp                PASS -> 7


In [16]:
rng = random.Random(594)
for _ in range(300):
    n = rng.randint(1, 10)
    nums = [rng.randint(-3, 3) for _ in range(n)]
    expected = lhs_bruteforce(nums)
    for mname, f in methods.items():
        assert f(nums) == expected, (nums, mname)
print("300 mảng ngẫu nhiên nhỏ: khớp brute force 2^n — PASS")

300 mảng ngẫu nhiên nhỏ: khớp brute force 2^n — PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Quên điều kiện "phải có cả `x+1`"
Dùng `count[x] + count.get(x+1, 0)` mà không kiểm tra → mảng `[1,1,1,1]` trả 4 thay vì 0.

In [17]:
def lhs_wrong_missing_check(nums):
    count = Counter(nums)
    return max(count[x] + count.get(x + 1, 0) for x in count)

print("Sai :", lhs_wrong_missing_check([1, 1, 1, 1]), "| Đúng:", lhs_counter([1, 1, 1, 1]))

Sai : 4 | Đúng: 0


### Lỗi 2: Nhầm subsequence thành subarray (đoạn liền nhau)
Tìm đoạn liền nhau dài nhất có max − min = 1 cho `[1,3,2,2,5,2,3,7]` được `[3,2,2]` dài 3 – sai, vì đề cho phép bỏ phần tử ở giữa.

### Lỗi 3: Xét cả `x−1` và `x+1` nhưng dùng `count[x+1]` khi chưa kiểm tra (với dict thường)
`dict` thường sẽ `KeyError`; `Counter` thì trả 0 nhưng ta lại quên kiểm tra như Lỗi 1.

### Lỗi 4: Cửa sổ trượt cập nhật `best` khi chênh = 0
Viết `if arr[right] - arr[left] <= 1:` → `[1,1,1,1]` ra 4. Phải là `== 1`.

In [18]:
def longest_subarray_wrong(nums):
    """Hiểu nhầm thành subarray liền nhau (brute force O(n^2))."""
    best = 0
    for i in range(len(nums)):
        for j in range(i, len(nums)):
            seg = nums[i:j + 1]
            if max(seg) - min(seg) == 1:
                best = max(best, len(seg))
    return best

def window_wrong(nums):
    arr, best, left = sorted(nums), 0, 0
    for right in range(len(arr)):
        while arr[right] - arr[left] > 1:
            left += 1
        if arr[right] - arr[left] <= 1:     # SAI: chấp nhận chênh 0
            best = max(best, right - left + 1)
    return best

print("Hiểu nhầm subarray:", longest_subarray_wrong([1, 3, 2, 2, 5, 2, 3, 7]), "(đúng là 5)")
print("Cửa sổ <= 1        :", window_wrong([1, 1, 1, 1]), "(đúng là 0)")

Hiểu nhầm subarray: 3 (đúng là 5)
Cửa sổ <= 1        : 4 (đúng là 0)


## 12. Mẹo nhận dạng pattern

- Đề nói **subsequence** nhưng điều kiện chỉ liên quan tới **giá trị** (max/min, tổng, có mặt) chứ không liên quan vị trí → thường quy về **đếm tần suất**.
- "Hai giá trị cách nhau đúng `k`" → `Counter` + tra `x + k in count`.
- Nếu không muốn dùng hash: **sort** rồi xét đoạn liên tiếp / hai con trỏ.
- Mẫu "với mỗi phần tử, tìm bạn đôi" → giống Two Sum: lưu vào dict rồi tra cứu.

## 13. Biến thể

**Biến thể A — max − min ≤ 1** (cho phép toàn số giống nhau): bỏ điều kiện bắt buộc có `x+1`.

**Biến thể B — chênh lệch đúng `k`**: tra `x + k`. Cẩn thận `k = 0`: khi đó dãy chỉ gồm một giá trị, đáp án = `max(count)`.

**Biến thể C — subarray liền nhau**: phải dùng cửa sổ trượt trên mảng **gốc** (không sort).

In [19]:
def lhs_at_most_one(nums):
    count = Counter(nums)
    return max(count[x] + count.get(x + 1, 0) for x in count)

def lhs_diff_k(nums, k):
    count = Counter(nums)
    if k == 0:
        return max(count.values())
    best = 0
    for x in count:
        if x + k in count:
            best = max(best, count[x] + count[x + k])
    return best

print("A:", lhs_at_most_one([1, 1, 1, 1]), lhs_at_most_one([1, 3, 2, 2, 5, 2, 3, 7]))
print("B (k=2):", lhs_diff_k([1, 3, 2, 2, 5, 2, 3, 7], 2), "| B (k=0):", lhs_diff_k([1, 1, 2], 0))

A: 4 5
B (k=2): 3 | B (k=0): 2


## 14. Bài tập (TODO – không gọi khi chưa làm)

**Bài 1 — Đoán output.** `Counter([4, 4, 5, 9])[6]` in ra gì? Sau dòng đó, `6 in Counter(...)` là gì?

**Bài 2 — Tính tay.** `nums = [2, 2, 3, 1, 1, 1, 4, 3, 3]`. Lập bảng `count`, tính đáp án.

**Bài 3 — Điền chỗ trống:** viết `findLHS` bằng `dict` thường (không dùng `Counter`).

**Bài 4 — Trả về chính dãy con** hài hòa dài nhất (giữ thứ tự gốc). Nếu nhiều đáp án, chọn cặp `x` nhỏ nhất. Nếu không có, trả `[]`.

**Bài 5 — Biến thể:** đếm **số cặp giá trị** `(x, x+1)` cùng xuất hiện.

**Bài 6 — Mini challenge:** độ dài **subarray liền nhau** dài nhất có max − min đúng bằng 1, trong `O(n)` (hoặc gần `O(n)`).

In [20]:
answer_ex1 = None      # TODO: (giá trị in ra, True/False)
answer_ex2 = None      # TODO: số nguyên


def lhs_plain_dict(nums):
    # TODO 1: tạo count = {} và đếm bằng count.get(v, 0) + 1
    # TODO 2: duyệt key, kiểm tra x + 1 in count
    pass


def lhs_sequence(nums):
    # TODO: tìm x tốt nhất rồi lọc nums giữ các phần tử bằng x hoặc x+1
    pass

# Kiểm tra sau khi làm:
# assert lhs_plain_dict([1, 3, 2, 2, 5, 2, 3, 7]) == 5
# assert lhs_sequence([1, 3, 2, 2, 5, 2, 3, 7]) == [3, 2, 2, 2, 3]

In [21]:
def count_harmonious_pairs(nums):
    # TODO: đếm số x khác nhau sao cho x+1 cũng có mặt
    pass


def longest_harmonious_subarray(nums):
    # TODO: cửa sổ trượt trên mảng GỐC, dict đếm phần tử trong cửa sổ
    pass

# Kiểm tra sau khi làm:
# assert count_harmonious_pairs([1, 2, 3, 5, 6]) == 3
# assert longest_harmonious_subarray([1, 3, 2, 2, 5, 2, 3, 7]) == 3

## 15. Gợi ý 3 mức

**Bài 1:** (1) Counter trả 0 cho key thiếu. (2) Đọc `c[k]` không thêm key. (3) Thử in `c` sau khi đọc.

**Bài 2:** (1) Đếm từng giá trị. (2) Xét các cặp `(1,2)`, `(2,3)`, `(3,4)`. (3) So sánh 3 tổng.

**Bài 3:** (1) Giống Counter nhưng tự đếm. (2) `count[v] = count.get(v, 0) + 1`. (3) Vòng 2 giống hệt `lhs_counter`.

**Bài 4:** (1) Tìm `x` cho tổng lớn nhất trước. (2) Duyệt `sorted(count)` để ưu tiên `x` nhỏ, dùng `>` (không `>=`). (3) `[v for v in nums if v == x or v == x + 1]`.

**Bài 5:** (1) Không cần tổng, chỉ cần đếm. (2) `sum(1 for x in count if x + 1 in count)`. (3) Một dòng.

**Bài 6:** (1) Cửa sổ `[left, right]` dài nhất có max − min ≤ 1 kết thúc tại `right`. (2) Giữ `dict` đếm phần tử trong cửa sổ; khi `max(keys) − min(keys) > 1` thì bỏ `nums[left]`, tăng `left` (dict có tối đa 3 key nên `max/min` là `O(1)`). (3) Chỉ cập nhật đáp án khi `max − min == 1`.

## 16. Đáp án tham khảo (chạy kiểm tra)

In [22]:
c_demo = Counter([4, 4, 5, 9])
ans_ex1 = (c_demo[6], 6 in c_demo)          # (0, False)
ans_ex2 = lhs_counter([2, 2, 3, 1, 1, 1, 4, 3, 3])   # cặp (2,3): 2 + 3 = 5


def ans_lhs_plain_dict(nums):
    count = {}
    for v in nums:
        count[v] = count.get(v, 0) + 1
    best = 0
    for x in count:
        if x + 1 in count:
            best = max(best, count[x] + count[x + 1])
    return best


def ans_lhs_sequence(nums):
    count = Counter(nums)
    best, best_x = 0, None
    for x in sorted(count):
        if x + 1 in count and count[x] + count[x + 1] > best:
            best, best_x = count[x] + count[x + 1], x
    if best_x is None:
        return []
    return [v for v in nums if v == best_x or v == best_x + 1]


def ans_count_harmonious_pairs(nums):
    count = Counter(nums)
    return sum(1 for x in count if x + 1 in count)


def ans_longest_harmonious_subarray(nums):
    window = {}
    left = best = 0
    for right, v in enumerate(nums):
        window[v] = window.get(v, 0) + 1
        while max(window) - min(window) > 1:
            out = nums[left]
            window[out] -= 1
            if window[out] == 0:
                del window[out]
            left += 1
        if max(window) - min(window) == 1:
            best = max(best, right - left + 1)
    return best


print("Bài 1:", ans_ex1, "| Bài 2:", ans_ex2)
assert ans_ex1 == (0, False) and ans_ex2 == 5
assert ans_lhs_plain_dict([1, 3, 2, 2, 5, 2, 3, 7]) == 5
assert ans_lhs_plain_dict([1, 1, 1, 1]) == 0
assert ans_lhs_sequence([1, 3, 2, 2, 5, 2, 3, 7]) == [3, 2, 2, 2, 3]
assert ans_lhs_sequence([1, 2, 3, 4]) == [1, 2]
assert ans_lhs_sequence([7, 7]) == []
assert ans_count_harmonious_pairs([1, 2, 3, 5, 6]) == 3
assert ans_longest_harmonious_subarray([1, 3, 2, 2, 5, 2, 3, 7]) == 3
assert ans_longest_harmonious_subarray([1, 1, 1]) == 0

rng = random.Random(6)
for _ in range(300):
    nums = [rng.randint(0, 4) for _ in range(rng.randint(1, 12))]
    assert ans_longest_harmonious_subarray(nums) == longest_subarray_wrong(nums)  # so với brute force subarray
    assert ans_lhs_plain_dict(nums) == lhs_bruteforce(nums)
print("Đáp án: tất cả test PASS")

Bài 1: (0, False) | Bài 2: 5


Đáp án: tất cả test PASS


## 17. Template nhớ nhanh và checklist submit

```python
from collections import Counter

class Solution:
    def findLHS(self, nums: list[int]) -> int:
        count = Counter(nums)
        best = 0
        for x in count:
            if x + 1 in count:                 # phải có CẢ x và x+1
                best = max(best, count[x] + count[x + 1])
        return best
```

Tư duy 3 bước: **subsequence + điều kiện theo giá trị → bỏ qua thứ tự → đếm tần suất → tra bạn đôi `x+1`**.

### Checklist
- [ ] "Đúng bằng 1" chứ không phải "≤ 1" → mảng toàn giống nhau trả `0`?
- [ ] Kiểm tra `x + 1 in count` trước khi cộng?
- [ ] Không nhầm subsequence thành subarray?
- [ ] Số âm và số rất lớn (`±10^9`) vẫn đúng (Python int không tràn)?
- [ ] Mảng 1 phần tử trả `0`?